# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamzanPUCIT/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

---
**Lane: Refresh / Content Opportunity Scoring** (core lane 2).

Why this one: it is the lane closest to a real decision a person makes every week — *which pages
should an editor open first?* The starter dataset already has the two windows I need
(`impressions_prev_30d` = days 31–60 back, `impressions_last_30d` = the most recent 30 days), so I
can define an outcome that is **observed later** than the inputs, instead of copying a rule.
It also leads naturally into a ranked queue with reason codes, which is the final output of the track.

In [1]:
# Setup: load the starter dataset (works in a local clone and in Colab)
from pathlib import Path
import numpy as np
import pandas as pd

RAW_URL = ("https://raw.githubusercontent.com/RamzanPUCIT/flyrank-ml-internship/"
           "main/data/raw/content_refresh_anonymized.csv")
local = [p for p in [Path("../../data/raw/content_refresh_anonymized.csv"),
                     Path("data/raw/content_refresh_anonymized.csv")] if p.exists()]
df = pd.read_csv(local[0] if local else RAW_URL)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("rows x cols:", df.shape)

print("pages:", df["content_id"].nunique(), "| clients:", df["client_id"].nunique())
df["content_type"].value_counts()

rows x cols: (30000, 44)
pages: 30000 | clients: 32


content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

---
- **Decision it improves:** which content pages an editor should review first for a refresh
  (and which can wait / just be monitored).
- **Who acts:** the content editor or SEO manager of one client site. They take the top of a
  ranked list and refresh, expand, or rewrite the title/meta of those pages.
- **Cost of a wrong call:**
  - *False alarm* → an editor spends 30–60 minutes on a page that was fine.
  - *Miss* → a page that had real search demand keeps losing impressions and nobody looks at it.
  - A miss on a high-demand page costs more than one wasted review, so the queue should put
    pages with real demand near the top, not just any page that dropped.
- The list matters because a client has a few hundred pages (median shown below) — nobody
  can review all of them.

In [2]:
pages_per_client = df.groupby("client_id").size()
print("pages per client  -> median:", int(pages_per_client.median()),
      "| min:", pages_per_client.min(), "| max:", pages_per_client.max())

pages per client  -> median: 567 | min: 3 | max: 7008


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

---
Three numbers from the cell below that make this lane worth the next weeks:

1. In my slice (pages with ≥ 100 impressions in the earlier 30-day window) about **6 in 10 pages**
   had their impressions fall by more than 20% in the most recent 30 days.
2. Those dropping pages lost roughly **a quarter of all earlier-window impressions** in the slice.
3. About **38%** of slice pages got **zero clicks** in the earlier window despite ≥ 100
   impressions — a visible, cheap-to-act-on signal (title/meta).

In [3]:
# My lane slice (same definition in every notebook)
MIN_PREV_IMPR = 100          # page needs real demand in the earlier window
lane = df[df["impressions_prev_30d"] >= MIN_PREV_IMPR].copy()
# proxy target: observed later-window outcome (last 30d vs the 30d before)
lane["demand_drop"] = (lane["impressions_last_30d"] < 0.8 * lane["impressions_prev_30d"]).astype(int)

drop = lane[lane["demand_drop"] == 1]
lost = (drop["impressions_prev_30d"] - drop["impressions_last_30d"]).sum()
print(f"slice pages: {len(lane):,} from {lane['client_id'].nunique()} clients")
print(f"share with >20% impression drop: {lane['demand_drop'].mean():.1%}")
print(f"impressions lost by dropping pages: {lost:,.0f} "
      f"({lost / lane['impressions_prev_30d'].sum():.1%} of earlier-window impressions)")
print(f"zero-click pages in the earlier window: {(lane['clicks_prev_30d'] == 0).mean():.1%}")

slice pages: 18,010 from 30 clients
share with >20% impression drop: 61.6%
impressions lost by dropping pages: 13,991,893 (26.3% of earlier-window impressions)
zero-click pages in the earlier window: 38.4%


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

---
**What my work can say:**
- *observed* / *measured*: e.g. "in this 30k-page slice, X% of pages with demand dropped".
- *directional*: "younger pages and zero-click pages dropped more often in this data".
- *decision-support*: "this ranked list is a better starting point for an editor than random order
  or alphabetical order".

**What it will never say:**
- that a refresh *causes* recovery (no experiment here, just observation);
- anything about "how Google ranks" or "predicting Google";
- anything about a specific client — IDs are pseudonyms and are only used for grouping/splits.

The label source columns (`trend_direction`, `trend_pct`) and anything from the last-30-day window
are never model inputs — the check below keeps me honest.

In [4]:
LABEL_WINDOW_COLS = ["trend_direction", "trend_pct", "impressions_last_30d",
                     "clicks_last_30d", "sessions_last_30d"]
planned_inputs = ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
                  "content_age_days", "search_volume", "competition", "cpc", "word_count",
                  "content_type", "main_intent"]
assert not set(planned_inputs) & set(LABEL_WINDOW_COLS), "label-window column in inputs!"
print("OK - no label-window columns in my planned inputs:", planned_inputs)

OK - no label-window columns in my planned inputs: ['impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'search_volume', 'competition', 'cpc', 'word_count', 'content_type', 'main_intent']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.